# Week 7: Task 4 - Iris Flower Classification
## Exploratory Data Visualization (Tuesday Deliverable)

**Internship:** Arch Technologies, Machine Learning Domain, Month 2  
**Task:** Task 4 — Iris Flower Classification (First Half)  
**Author:** Sharjeel Shahzad  

### Overview & Objectives
Following initial data loading and exploration on Monday, the Tuesday deliverable focuses on **Exploratory Data Visualization**:
1. Creating comprehensive **pair plots** showing pairwise interactions and KDE distributions across all 4 morphological features colored by species.
2. Generating high-resolution **scatter plots** highlighting class separability (Petal Length vs. Petal Width and Sepal Length vs. Sepal Width).
3. Analyzing feature distributions using box plots and violin plots to assess spread and variance.
4. Computing and visualizing the feature correlation matrix.
5. Exporting key production charts to:
   - `outputs/pairplot.png`
   - `outputs/scatterplot.png`


## 1. Environment Setup & Data Loading


In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Visual formatting configurations
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 10

# Paths
data_path = Path("../data/iris-dataset.csv")
if not data_path.exists():
    data_path = Path("week-7/data/iris-dataset.csv")

outputs_dir = Path("../outputs")
if not outputs_dir.exists():
    outputs_dir = Path("week-7/outputs")
outputs_dir.mkdir(parents=True, exist_ok=True)

# Load data and drop Id
df = pd.read_csv(data_path)
if "Id" in df.columns:
    df.drop(columns=["Id"], inplace=True)

print(f"Data ready for visualization: {df.shape[0]} samples, {df.shape[1]} columns.")
palette = {'Iris-setosa': '#3b82f6', 'Iris-versicolor': '#10b981', 'Iris-virginica': '#8b5cf6'}


Data ready for visualization: 150 samples, 5 columns.


## 2. Multi-Feature Pair Plot Analysis
We generate a full 4x4 pairwise grid displaying bivariate scatter relationships off the diagonal and univariate Kernel Density Estimates (KDE) along the diagonal.
This visualization is exported as `outputs/pairplot.png`.


In [2]:
# Generate Pairplot
g = sns.pairplot(
    df,
    hue="Species",
    palette=palette,
    markers=["o", "s", "D"],
    diag_kind="kde",
    height=2.5,
    plot_kws={"alpha": 0.8, "s": 50},
    diag_kws={"fill": True, "alpha": 0.4}
)
g.fig.suptitle("Iris Species Pairwise Feature Relationships & Distributions", y=1.02, fontsize=14, fontweight="bold")

# Export high-resolution chart
pairplot_path = outputs_dir / "pairplot.png"
g.savefig(pairplot_path, dpi=300, bbox_inches="tight")
print(f"Pair plot exported to: {pairplot_path}")
plt.show()


Pair plot exported to: ../outputs/pairplot.png


### Pair Plot Observations:
- **Setosa Separability**: `Iris-setosa` (blue) forms a distinct, isolated cluster in every subplot involving petal length or petal width. It is completely linearly separable.
- **Versicolor vs. Virginica**: `Iris-versicolor` (green) and `Iris-virginica` (purple) exhibit clear trend differences but touch each other near `PetalLength ~ 4.8 cm` and `PetalWidth ~ 1.7 cm`.
- **KDE Diagonals**: The diagonal distribution curves show unimodal bimodal/trimodal patterns. `PetalLengthCm` and `PetalWidthCm` show almost zero overlap between Setosa and the other two classes.


## 3. Class Separability Scatter Plots
We construct side-by-side scatter plots highlighting:
1. **Petal Dimensions**: The primary driver of species separability.
2. **Sepal Dimensions**: Showing wider dispersion and partial morphological overlap.
Exported as `outputs/scatterplot.png`.


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Panel 1: Petal Length vs Petal Width
sns.scatterplot(
    data=df, x="PetalLengthCm", y="PetalWidthCm", hue="Species",
    style="Species", palette=palette, s=90, alpha=0.9, ax=axes[0]
)
axes[0].set_title("Petal Length vs. Petal Width (High Separability)", fontsize=12, fontweight="bold", pad=12)
axes[0].set_xlabel("Petal Length (cm)", fontsize=11, fontweight="semibold")
axes[0].set_ylabel("Petal Width (cm)", fontsize=11, fontweight="semibold")
axes[0].axvline(x=2.5, color="red", linestyle="--", alpha=0.6, label="Setosa Linear Boundary")
axes[0].legend(loc="upper left")

# Panel 2: Sepal Length vs Sepal Width
sns.scatterplot(
    data=df, x="SepalLengthCm", y="SepalWidthCm", hue="Species",
    style="Species", palette=palette, s=90, alpha=0.9, ax=axes[1]
)
axes[1].set_title("Sepal Length vs. Sepal Width (Morphological Overlap)", fontsize=12, fontweight="bold", pad=12)
axes[1].set_xlabel("Sepal Length (cm)", fontsize=11, fontweight="semibold")
axes[1].set_ylabel("Sepal Width (cm)", fontsize=11, fontweight="semibold")
axes[1].legend(loc="upper right")

plt.suptitle("Iris Flower Classification: Class Separability Analysis", fontsize=15, fontweight="bold", y=1.0)
plt.tight_layout()

# Export chart
scatterplot_path = outputs_dir / "scatterplot.png"
plt.savefig(scatterplot_path, dpi=300, bbox_inches="tight")
print(f"Scatter plots exported to: {scatterplot_path}")
plt.show()


Scatter plots exported to: ../outputs/scatterplot.png


## 4. Feature Distribution & Box Plot Analysis
Box plots allow us to examine median values, interquartile ranges, and initial outlier candidates across each species.


In [4]:
features = ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, feature in enumerate(features):
    row, col = idx // 2, idx % 2
    sns.boxplot(data=df, x="Species", y=feature, palette=palette, ax=axes[row, col], width=0.5)
    axes[row, col].set_title(f"Distribution of {feature} by Species", fontsize=12, fontweight="bold")
    axes[row, col].set_xlabel("")
    axes[row, col].set_ylabel(f"{feature} (cm)", fontsize=10)

plt.suptitle("Morphological Feature Distributions across Iris Species", fontsize=14, fontweight="bold", y=1.0)
plt.tight_layout()
plt.show()


## 5. Feature Correlation Heatmap
We compute Pearson correlation coefficients across the numerical features.


In [5]:
# Calculate Pearson correlation matrix
corr_matrix = df[features].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".3f",
    cmap="Blues",
    vmin=-1, vmax=1,
    square=True,
    linewidths=1.0,
    linecolor="white",
    cbar_kws={"shrink": 0.8}
)
plt.title("Pearson Correlation Matrix (Numerical Features)", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

corr_matrix


               SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm
SepalLengthCm       1.000000     -0.109369       0.871754      0.817954
SepalWidthCm       -0.109369      1.000000      -0.420516     -0.356544
PetalLengthCm       0.871754     -0.420516       1.000000      0.962757
PetalWidthCm        0.817954     -0.356544       0.962757      1.000000


## 6. Tuesday Visualization Summary & Insights

1. **Collinearity between Petals**: `PetalLengthCm` and `PetalWidthCm` exhibit an exceptionally strong positive correlation ($r = 0.963$). Both features increase synchronously from Setosa $\rightarrow$ Versicolor $\rightarrow$ Virginica.
2. **Linear Separability of Setosa**: A vertical threshold at `PetalLengthCm = 2.5 cm` separates 100% of Setosa samples with zero classification error.
3. **The Versicolor-Virginica Boundary**: While their median petal values differ significantly, their boundary exhibits mild density overlap. A linear model will need careful feature scaling and optimal hyperplane placement to partition them accurately.
4. **Outputs Generated**:
   - `outputs/pairplot.png`: Multi-panel pairplot capturing all feature combinations.
   - `outputs/scatterplot.png`: Side-by-side scatter plots highlighting class separability.
